# Week 3: Feature Engineering

## Objective
This phase focuses on creating time-based and lag features from the hourly
energy consumption dataset to prepare inputs for baseline and advanced
machine learning models.

In [1]:
import pandas as pd
import numpy as np

In [2]:
hourly_df = pd.read_csv(
    "../../data/processed/hourly_device_energy_scaled.csv",
    parse_dates=['timestamp']
)

print(hourly_df.head())
print(hourly_df.shape)

     Appliance Type           timestamp  Energy Consumption (kWh)
0  Air Conditioning 2023-01-01 00:00:00                  0.155360
1  Air Conditioning 2023-01-01 01:00:00                  0.267487
2  Air Conditioning 2023-01-01 02:00:00                  0.520211
3  Air Conditioning 2023-01-01 03:00:00                  0.147276
4  Air Conditioning 2023-01-01 04:00:00                  0.375395
(87828, 3)


In [3]:
hourly_df = hourly_df.set_index('timestamp')

# IMPORTANT: Sort index for correct lag creation
hourly_df = hourly_df.sort_index()

print(hourly_df.head())

              Appliance Type  Energy Consumption (kWh)
timestamp                                             
2023-01-01  Air Conditioning                  0.155360
2023-01-01        Dishwasher                  0.116344
2023-01-01   Washing Machine                  0.039367
2023-01-01            Fridge                  0.028120
2023-01-01         Microwave                  0.059051


In [4]:
hourly_df['hour'] = hourly_df.index.hour
hourly_df['day'] = hourly_df.index.day
hourly_df['weekday'] = hourly_df.index.weekday
hourly_df['month'] = hourly_df.index.month

hourly_df.head()

,Appliance Type,Energy Consumption (kWh),hour,day,weekday,month
timestamp,,,,,,
2023-01-01,Air Conditioning,0.155360,0,1,6,1
2023-01-01,Dishwasher,0.116344,0,1,6,1
2023-01-01,Washing Machine,0.039367,0,1,6,1
2023-01-01,Fridge,0.028120,0,1,6,1
2023-01-01,Microwave,0.059051,0,1,6,1


In [5]:
# %%
# Correct lag and rolling features per appliance (FIXED VERSION)

hourly_df['lag_1'] = (
    hourly_df.groupby('Appliance Type')['Energy Consumption (kWh)']
    .transform(lambda x: x.shift(1))
)

hourly_df['lag_24'] = (
    hourly_df.groupby('Appliance Type')['Energy Consumption (kWh)']
    .transform(lambda x: x.shift(24))
)

hourly_df['rolling_mean_24'] = (
    hourly_df.groupby('Appliance Type')['Energy Consumption (kWh)']
    .transform(lambda x: x.rolling(window=24).mean())
)

print(hourly_df.head(30))


                       Appliance Type  Energy Consumption (kWh)  hour  day  \
timestamp                                                                    
2023-01-01 00:00:00  Air Conditioning                  0.155360     0    1   
2023-01-01 00:00:00        Dishwasher                  0.116344     0    1   
2023-01-01 00:00:00   Washing Machine                  0.039367     0    1   
2023-01-01 00:00:00            Fridge                  0.028120     0    1   
2023-01-01 00:00:00         Microwave                  0.059051     0    1   
2023-01-01 00:00:00            Lights                  0.160633     0    1   
2023-01-01 01:00:00        Dishwasher                  0.132162     1    1   
2023-01-01 01:00:00   Washing Machine                  0.069596     1    1   
2023-01-01 01:00:00            Fridge                  0.024605     1    1   
2023-01-01 01:00:00                TV                  0.140949     1    1   
2023-01-01 01:00:00          Computer                  0.056239 

In [6]:
hourly_df = hourly_df.dropna()
hourly_df.shape

(87588, 9)

In [ ]:
appliances = hourly_df['Appliance Type'].unique()

print("Total appliances:", len(appliances))
print("Appliances:", appliances)

for appliance in appliances:
    appliance_df = hourly_df[
        hourly_df['Appliance Type'] == appliance
    ].copy()

    X = appliance_df.drop(
        ['Energy Consumption (kWh)', 'Appliance Type'],
        axis=1
    )
    y = appliance_df['Energy Consumption (kWh)']

    split_index = int(len(appliance_df) * 0.8)

    X_train = X.iloc[:split_index]
    X_test = X.iloc[split_index:]
    y_train = y.iloc[:split_index]
    y_test = y.iloc[split_index:]

    X_train.to_csv(f"../../data/model_inputs/X_train_{appliance}.csv", index=False)
    X_test.to_csv(f"../../data/model_inputs/X_test_{appliance}.csv", index=False)
    y_train.to_csv(f"../../data/model_inputs/y_train_{appliance}.csv", index=False)
    y_test.to_csv(f"../../data/model_inputs/y_test_{appliance}.csv", index=False)

print("Train-test datasets saved in data/model_inputs/")


Train-test datasets saved in data/model_inputs/
